# TDGraph-IDS -- Phase 3: Dynamic Sliding-Window Graph Topology

**Goal:** Model network traffic as a sequence of directed graphs across sliding
time windows and extract 9 topology features per node including 2 temporal delta
features that capture how each node's structural role changes between windows.

**Input:** `data/graph/graph_data.csv`

**Outputs:**
- `data/graph/topo_features_dynamic.csv` -- per-node topology across all windows
- `data/graph/topo_node_avg.csv` -- per-node averages for flow enrichment
- `data/graph/window_stats.csv` -- per-window graph statistics
- `data/processed/enriched_flows.csv` -- flows enriched with 9 topology features
- `data/graph/network_graph.gexf` -- full IP graph for Gephi
- `outputs/plots/phase3_*.png` -- submission visualizations

**Architecture (from report):**
- Window size: 5,000 flows, slide step: 2,500 (50% overlap)
- Nodes: unique IP addresses (not IP:Port)
- Edges: directed flows, weight = total bytes
- 7 base topology features per node per window
- 2 temporal delta features: degree_delta and volume_delta
- EWMA warm-start (alpha=0.1) for new IP nodes
- 2-sigma anomaly detection rule on degree_delta

**Research novelty:**
Prior work (E-GraphSAGE, TE-G-SAGE) uses a single static graph or independent
per-window snapshots with no cross-window signal. TDGraph-IDS computes degree_delta
and volume_delta across consecutive windows -- the rate-of-change of topology
is the intrusion signal no prior work uses:

- `degree_delta` -- out-degree spike signals port scan or lateral movement onset
- `volume_delta` -- traffic volume spike signals DoS or exfiltration onset

---

## 3.1 -- Setup

In [1]:
import sys
import os
import time
import warnings
import logging
from collections import defaultdict

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker

warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config import CFG, ensure_dirs

ensure_dirs()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('phase3')

# Window parameters (from report)
WINDOW_SIZE  = CFG.WINDOW_SIZE   # 5000 flows per window
SLIDE_STEP   = CFG.SLIDE_STEP    # 2500 flows (50% overlap)
MAX_WINDOWS  = CFG.MAX_WINDOWS   # 50 windows max
BC_SAMPLE_K  = CFG.BC_SAMPLE_K   # betweenness centrality approximation
EWMA_ALPHA   = 0.1               # slow adaptation as specified in report

log.info('Project root : %s', CFG.ROOT)
log.info('Window size  : %d flows', WINDOW_SIZE)
log.info('Slide step   : %d flows (%.0f%% overlap)',
         SLIDE_STEP, (1 - SLIDE_STEP/WINDOW_SIZE)*100)
log.info('Max windows  : %d', MAX_WINDOWS)
log.info('EWMA alpha   : %.1f', EWMA_ALPHA)

08:46:04  INFO  Project root : C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS
08:46:04  INFO  Window size  : 5000 flows
08:46:04  INFO  Slide step   : 2500 flows (50% overlap)
08:46:04  INFO  Max windows  : 50
08:46:04  INFO  EWMA alpha   : 0.1


## 3.2 -- Load graph data

Load the IP-level flow data saved by Phase 2. Validate required columns,
coerce numeric types, and confirm IP addresses are present.

Note: CIC-IDS-2018 strips IP addresses so graph construction uses
NF-UNSW-NB15-v2 flows only. This is acknowledged in the report as a
dataset constraint, not a methodological choice.

In [3]:
def load_graph_data(path: str) -> pd.DataFrame:
    """
    Load graph_data.csv produced by Phase 2 and validate structure.

    Parameters
    ----------
    path : str
        Path to graph_data.csv.

    Returns
    -------
    pd.DataFrame
        Validated graph data with correct dtypes.

    Raises
    ------
    FileNotFoundError
        If graph_data.csv does not exist.
    ValueError
        If required columns are absent or no IP address data found.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'graph_data.csv not found at {path}. Run Phase 2 first.'
        )

    log.info('Loading graph_data.csv ...')
    df = pd.read_csv(path, low_memory=False)
    log.info('  Raw shape: %d rows x %d columns', *df.shape)

    required = {'src_ip', 'dst_ip', 'fwd_bytes', 'fwd_pkts', 'label'}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(
            f'Required columns missing: {missing}. Re-run Phase 2.'
        )

    # Coerce numeric columns
    for col in ['fwd_bytes', 'fwd_pkts', 'bwd_bytes', 'bwd_pkts',
                'duration_ms', 'label']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    df['label'] = df['label'].astype(int)

    # Graph uses only flows with valid IP addresses
    # CIC-IDS-2018 has no IPs -- filter to NF-UNSW rows only for graph
    has_ip = (
        df['src_ip'].notna() &
        df['dst_ip'].notna() &
        (df['src_ip'].astype(str) != 'nan') &
        (df['dst_ip'].astype(str) != 'nan')
    )
    graph_df = df[has_ip].copy().reset_index(drop=True)

    if len(graph_df) == 0:
        raise ValueError(
            'No rows with valid IP addresses found. '
            'Graph construction requires NF-UNSW-NB15-v2 flows.'
        )

    log.info('  Rows with valid IPs : %d / %d', len(graph_df), len(df))
    log.info('  Unique src IPs      : %d', graph_df['src_ip'].nunique())
    log.info('  Unique dst IPs      : %d', graph_df['dst_ip'].nunique())
    log.info('  Attack flows        : %d (%.2f%%)',
             graph_df['label'].sum(), graph_df['label'].mean() * 100)

    return graph_df


graph_df = load_graph_data(str(CFG.DATA_GRAPH / 'graph_data.csv'))

total_unique_ips = pd.concat(
    [graph_df['src_ip'], graph_df['dst_ip']]
).nunique()

print('Shape            :', graph_df.shape)
print('Unique src IPs   :', graph_df['src_ip'].nunique())
print('Unique dst IPs   :', graph_df['dst_ip'].nunique())
print('Total unique IPs :', total_unique_ips)
print('Attack rate      : {:.2f}%'.format(graph_df['label'].mean() * 100))
print()
print('Source breakdown (NF-UNSW vs CIC):')
if 'source' in graph_df.columns:
    print(graph_df['source'].value_counts().to_string())

08:46:52  INFO  Loading graph_data.csv ...
08:46:57  INFO    Raw shape: 2365636 rows x 13 columns
08:46:58  INFO    Rows with valid IPs : 2365636 / 2365636
08:46:58  INFO    Unique src IPs      : 40
08:46:58  INFO    Unique dst IPs      : 40
08:46:58  INFO    Attack flows        : 82977 (3.51%)


Shape            : (2365636, 13)
Unique src IPs   : 40
Unique dst IPs   : 40
Total unique IPs : 44
Attack rate      : 3.51%

Source breakdown (NF-UNSW vs CIC):
source
NF-UNSW    2365636


## 3.3 -- EWMA warm-start for new nodes

In a streaming context, new IP addresses appear in every window.
Without warm-start, a new node gets zero values for all topology features,
appearing as quiet and benign regardless of its subnet.

The EWMA warm-start (alpha=0.1) borrows subnet-level estimates for new nodes:
- Identify the /24 subnet prefix of the new node (e.g., 59.166.0.x)
- Assign the subnet EWMA as the initial feature values
- Update with each observation: new = 0.1 x observed + 0.9 x previous

A new IP in the 59.166.x.x attacker subnet immediately inherits high
attack_ratio and elevated out_degree from its subnet siblings.

In [4]:
class EWMAWarmStart:
    """
    EWMA-based warm-start mechanism for new IP nodes.

    Maintains per-node and per-subnet EWMA feature profiles.
    New nodes borrow from their /24 subnet neighbours rather than
    starting from zero, eliminating cold-start vulnerability.

    Parameters
    ----------
    alpha : float
        EWMA smoothing factor. Report specifies 0.1 (slow adaptation).
        Low alpha ensures a single benign flow cannot rapidly lower
        a node's suspicion score.
    """

    FEATURE_NAMES = [
        'pagerank', 'betweenness', 'in_degree', 'out_degree',
        'clustering', 'flow_volume', 'attack_ratio',
        'degree_delta', 'volume_delta',
    ]

    def __init__(self, alpha: float = 0.1):
        """
        Initialise EWMA warm-start with given smoothing factor.

        Parameters
        ----------
        alpha : float
            EWMA smoothing factor. Report specifies 0.1.
        """
        self.alpha        = alpha
        self.node_stats   = {}   # node -> feature dict
        self.subnet_stats = defaultdict(lambda: defaultdict(list))

    def _get_subnet(self, ip: str) -> str:
        """
        Extract the /24 subnet prefix from an IP address.

        Parameters
        ----------
        ip : str  IP address string.

        Returns
        -------
        str  /24 subnet prefix (e.g. '59.166.0').
        """
        parts = str(ip).split('.')
        return '.'.join(parts[:3]) if len(parts) >= 3 else 'unknown'

    def update(self, node: str, features: dict) -> None:
        """
        Update EWMA statistics for a node and its subnet.

        Parameters
        ----------
        node     : str   IP address string.
        features : dict  Feature name -> value mapping.
        """
        subnet = self._get_subnet(node)

        if node not in self.node_stats:
            self.node_stats[node] = features.copy()
        else:
            for feat, val in features.items():
                old = self.node_stats[node].get(feat, val)
                self.node_stats[node][feat] = (
                    self.alpha * val + (1 - self.alpha) * old
                )

        for feat, val in features.items():
            self.subnet_stats[subnet][feat].append(val)

    def get_initial_features(self, node: str) -> dict:
        """
        Get initial feature values for a new node.

        Returns subnet-level mean if subnet is known,
        otherwise returns zeros.

        Parameters
        ----------
        node : str  IP address of the new node.

        Returns
        -------
        dict  Feature name -> initial value mapping.
        """
        subnet = self._get_subnet(node)
        if subnet in self.subnet_stats:
            return {
                feat: float(np.mean(vals))
                for feat, vals in self.subnet_stats[subnet].items()
            }
        return {feat: 0.0 for feat in self.FEATURE_NAMES}


warm_start = EWMAWarmStart(alpha=EWMA_ALPHA)
print('EWMAWarmStart initialised with alpha={}'.format(EWMA_ALPHA))

EWMAWarmStart initialised with alpha=0.1


## 3.4 -- Per-window graph construction and feature extraction

For each window:
1. Build directed graph G where each unique IP is a node
2. Each flow is a directed edge from src_ip to dst_ip, weight = total bytes
3. Extract 7 base topology features per node
4. Compute 2 delta features vs previous window
5. Apply 2-sigma anomaly rule on degree_delta
6. Update EWMA warm-start with current window observations

In [5]:
def build_window_graph(window_df: pd.DataFrame) -> nx.DiGraph:
    """
    Build a directed weighted graph from one window of flows.

    Nodes are unique IP addresses. Each (src_ip, dst_ip) pair
    is aggregated into a single weighted edge.

    Parameters
    ----------
    window_df : pd.DataFrame
        One window of flow records with src_ip, dst_ip, fwd_bytes, label.

    Returns
    -------
    nx.DiGraph
        Directed weighted graph for this window.
    """
    agg = (
        window_df.groupby(['src_ip', 'dst_ip'])
        .agg(
            total_bytes  = ('fwd_bytes', 'sum'),
            flow_count   = ('fwd_bytes', 'count'),
            attack_flows = ('label',     'sum'),
        )
        .reset_index()
    )

    G = nx.DiGraph()
    for _, row in agg.iterrows():
        G.add_edge(
            str(row['src_ip']),
            str(row['dst_ip']),
            weight       = float(row['total_bytes']) + 1.0,
            flow_count   = int(row['flow_count']),
            attack_flows = int(row['attack_flows']),
            is_malicious = int(row['attack_flows'] > 0),
        )
    return G


def extract_window_features(
    G: nx.DiGraph,
    win_id: int,
    prev_features: dict,
    warm_start: EWMAWarmStart,
) -> tuple:
    """
    Extract 9 topology features per node for one window.

    Features (as specified in the report):
        pagerank      -- weighted PageRank
        betweenness   -- sampled betweenness centrality (k=100)
        in_degree     -- count of incoming edges
        out_degree    -- count of outgoing edges
        clustering    -- local clustering coefficient
        flow_volume   -- total bytes (in + out edge weights)
        attack_ratio  -- malicious flows / total flows at node
        degree_delta  -- |out_degree[t] - out_degree[t-1]|
        volume_delta  -- |flow_volume[t] - flow_volume[t-1]|

    Parameters
    ----------
    G             : nx.DiGraph      Current window graph.
    win_id        : int             Window index.
    prev_features : dict            node -> feature dict from previous window.
    warm_start    : EWMAWarmStart   Warm-start manager.

    Returns
    -------
    tuple of (topo_df, updated_prev_features)
        topo_df : pd.DataFrame  Per-node features for this window.
        updated : dict          Updated previous features for next window.
    """
    if G.number_of_nodes() == 0:
        return pd.DataFrame(), prev_features

    nodes = list(G.nodes())
    n     = len(nodes)

    # PageRank
    try:
        pr = nx.pagerank(G, weight='weight', alpha=0.85,
                         max_iter=100, tol=1e-4)
    except Exception:
        pr = {nd: 1.0 / n for nd in nodes}

    # Betweenness centrality (sampled for performance)
    k = min(BC_SAMPLE_K, n)
    try:
        bc = nx.betweenness_centrality(
            G, k=k, normalized=True, weight='weight'
        )
    except Exception:
        bc = {nd: 0.0 for nd in nodes}

    # Clustering coefficient
    try:
        cl = nx.clustering(G.to_undirected())
    except Exception:
        cl = {nd: 0.0 for nd in nodes}

    # Degree and volume features
    in_deg  = dict(G.in_degree())
    out_deg = dict(G.out_degree())

    records = []
    updated_prev = {}

    for nd in nodes:
        out_edges = list(G.out_edges(nd, data=True))
        in_edges  = list(G.in_edges(nd,  data=True))

        out_vol = sum(d.get('weight', 0) for _, _, d in out_edges)
        in_vol  = sum(d.get('weight', 0) for _, _, d in in_edges)
        volume  = out_vol + in_vol

        total_flows = sum(d.get('flow_count', 1) for _, _, d in out_edges)
        attack_fl   = sum(d.get('attack_flows', 0) for _, _, d in out_edges)
        attack_ratio = attack_fl / max(total_flows, 1)

        # Warm-start: use subnet estimates for new nodes
        if nd not in prev_features:
            initial = warm_start.get_initial_features(nd)
            prev_out_deg = initial.get('out_degree', 0.0)
            prev_volume  = initial.get('flow_volume', 0.0)
        else:
            prev_out_deg = prev_features[nd].get('out_degree', 0.0)
            prev_volume  = prev_features[nd].get('flow_volume', 0.0)

        # Delta features (core novel contribution)
        degree_delta = abs(out_deg.get(nd, 0) - prev_out_deg)
        volume_delta = abs(volume - prev_volume)

        feat = {
            'node'         : nd,
            'window_id'    : win_id,
            'pagerank'     : pr.get(nd, 0.0),
            'betweenness'  : bc.get(nd, 0.0),
            'in_degree'    : in_deg.get(nd, 0),
            'out_degree'   : out_deg.get(nd, 0),
            'clustering'   : cl.get(nd, 0.0),
            'flow_volume'  : volume,
            'attack_ratio' : attack_ratio,
            'degree_delta' : degree_delta,
            'volume_delta' : volume_delta,
        }
        records.append(feat)
        updated_prev[nd] = feat

        # Update EWMA warm-start
        warm_start.update(nd, {
            'pagerank'     : feat['pagerank'],
            'betweenness'  : feat['betweenness'],
            'in_degree'    : feat['in_degree'],
            'out_degree'   : feat['out_degree'],
            'clustering'   : feat['clustering'],
            'flow_volume'  : feat['flow_volume'],
            'attack_ratio' : feat['attack_ratio'],
            'degree_delta' : feat['degree_delta'],
            'volume_delta' : feat['volume_delta'],
        })

    return pd.DataFrame(records), updated_prev


print('Graph construction functions defined')

Graph construction functions defined


## 3.5 -- 2-sigma topology anomaly detection

Independent of the ML pipeline, the graph engine performs rule-based
anomaly detection based purely on topology dynamics.

**Rule (from report):** If a node's degree_delta exceeds 2 standard
deviations above its subnet mean degree_delta, flag as topology anomaly.

This independently detects port scan onset (sudden out-degree spike),
DDoS onset (sudden in-degree spike at victim), and botnet activation
(coordinated out-degree spikes across multiple subnet nodes).

In [6]:
def apply_anomaly_detection(
    topo_df: pd.DataFrame,
    warm_start: EWMAWarmStart,
) -> pd.DataFrame:
    """
    Apply the 2-sigma anomaly detection rule to topology features.

    Rule: flag a node if its degree_delta exceeds 2 standard deviations
    above the subnet mean degree_delta. This captures statistically
    unusual out-degree growth while maintaining low false positive rate.

    Parameters
    ----------
    topo_df    : pd.DataFrame    Per-node features for one window.
    warm_start : EWMAWarmStart   EWMA state for subnet-level baselines.

    Returns
    -------
    pd.DataFrame
        Input dataframe with 'topo_anomaly' column added (0 or 1).
    """
    topo_df = topo_df.copy()
    topo_df['topo_anomaly'] = 0

    for i, row in topo_df.iterrows():
        node   = str(row['node'])
        subnet = '.'.join(node.split('.')[:3])

        subnet_deltas = warm_start.subnet_stats.get(subnet, {}).get(
            'degree_delta', []
        )

        if len(subnet_deltas) >= 2:
            subnet_mean = np.mean(subnet_deltas)
            subnet_std  = np.std(subnet_deltas)
            threshold   = subnet_mean + 2 * subnet_std
            if row['degree_delta'] > max(threshold, 3):
                topo_df.at[i, 'topo_anomaly'] = 1
        else:
            # Insufficient subnet history -- flag high absolute deltas
            if row['degree_delta'] > 10:
                topo_df.at[i, 'topo_anomaly'] = 1

    return topo_df


print('Anomaly detection function defined')

Anomaly detection function defined


## 3.6 -- Run sliding window graph extraction

Process all 50 windows sequentially. For each window:
- Build directed graph from IP-level flows
- Extract 9 topology features per node
- Apply 2-sigma anomaly detection
- Update EWMA warm-start state
- Accumulate results and per-window statistics

Progress is logged every 10 windows.

In [7]:
def make_windows(
    df: pd.DataFrame,
    window_size: int,
    slide: int,
    max_windows: int,
) -> list:
    """
    Generate sliding window index ranges over the dataframe.

    50% overlap ensures every flow appears in at least two windows.
    An attack beginning near a window boundary is fully captured
    in the next window.

    Parameters
    ----------
    df          : pd.DataFrame  Full flow dataframe.
    window_size : int           Flows per window.
    slide       : int           Slide step between windows.
    max_windows : int           Maximum number of windows to process.

    Returns
    -------
    list of (window_id, start_idx, end_idx) tuples
    """
    windows = []
    start   = 0
    win_id  = 0
    while start + window_size <= len(df) and win_id < max_windows:
        windows.append((win_id, start, start + window_size))
        start  += slide
        win_id += 1
    return windows


windows = make_windows(graph_df, WINDOW_SIZE, SLIDE_STEP, MAX_WINDOWS)

print('Windows to process : {}'.format(len(windows)))
print('Flows per window   : {:,}'.format(WINDOW_SIZE))
print('Slide step         : {:,}'.format(SLIDE_STEP))
print('Overlap            : {:.0f}%'.format(
    (1 - SLIDE_STEP / WINDOW_SIZE) * 100
))
print('Total flows covered: {:,}'.format(len(graph_df)))

Windows to process : 50
Flows per window   : 5,000
Slide step         : 2,500
Overlap            : 50%
Total flows covered: 2,365,636


In [8]:
all_topo_records = []
window_stats     = []
prev_features    = {}   # node -> last window feature dict
t_start          = time.time()

log.info('Starting sliding window graph extraction ...')

for win_id, start, end in windows:

    win_df = graph_df.iloc[start:end]

    # Build graph
    G = build_window_graph(win_df)

    # Extract features
    topo_df, prev_features = extract_window_features(
        G, win_id, prev_features, warm_start
    )

    if topo_df.empty:
        log.warning('Window %d produced empty topology -- skipping', win_id)
        continue

    # Apply 2-sigma anomaly detection
    topo_df = apply_anomaly_detection(topo_df, warm_start)

    all_topo_records.append(topo_df)

    window_stats.append({
        'window_id'      : win_id,
        'start_flow'     : start,
        'end_flow'       : end,
        'n_flows'        : len(win_df),
        'n_nodes'        : G.number_of_nodes(),
        'n_edges'        : G.number_of_edges(),
        'density'        : round(nx.density(G), 6),
        'attack_rate'    : round(win_df['label'].mean(), 4),
        'topo_anomalies' : int(topo_df['topo_anomaly'].sum()),
        'elapsed_sec'    : round(time.time() - t_start, 2),
    })

    if win_id % 10 == 0 or win_id == 0:
        log.info(
            'Win %03d | nodes=%4d  edges=%5d  '
            'attack=%.3f  anomalies=%d  t=%.1fs',
            win_id,
            G.number_of_nodes(),
            G.number_of_edges(),
            win_df['label'].mean(),
            int(topo_df['topo_anomaly'].sum()),
            time.time() - t_start,
        )

topo_all   = pd.concat(all_topo_records, ignore_index=True)
stats_df   = pd.DataFrame(window_stats)
total_time = time.time() - t_start

log.info('Extraction complete in %.1f seconds', total_time)

print('\n--- Extraction Summary ---')
print('Windows processed    :', len(window_stats))
print('Topology records     :', len(topo_all))
print('Unique nodes tracked :', topo_all['node'].nunique())
print('Topo anomalies found :', int(topo_all['topo_anomaly'].sum()))
print('Total time           : {:.1f}s'.format(total_time))
print()
print('Topology feature columns:')
print([c for c in topo_all.columns if c != 'node'])

08:48:12  INFO  Starting sliding window graph extraction ...
08:48:13  INFO  Win 000 | nodes=  39  edges=  156  attack=0.001  anomalies=1  t=0.5s
08:48:13  INFO  Win 010 | nodes=  38  edges=  150  attack=0.059  anomalies=0  t=0.9s
08:48:13  INFO  Win 020 | nodes=  38  edges=  145  attack=0.046  anomalies=0  t=1.2s
08:48:14  INFO  Win 030 | nodes=  20  edges=  100  attack=0.000  anomalies=0  t=1.5s
08:48:14  INFO  Win 040 | nodes=  23  edges=  121  attack=0.000  anomalies=0  t=1.8s
08:48:14  INFO  Extraction complete in 2.2 seconds



--- Extraction Summary ---
Windows processed    : 50
Topology records     : 1557
Unique nodes tracked : 42
Topo anomalies found : 10
Total time           : 2.2s

Topology feature columns:
['window_id', 'pagerank', 'betweenness', 'in_degree', 'out_degree', 'clustering', 'flow_volume', 'attack_ratio', 'degree_delta', 'volume_delta', 'topo_anomaly']


## 3.7 -- Window statistics summary

Print per-window statistics to understand how graph topology
evolves across time and validate the delta features are capturing
meaningful variation.

In [9]:
print('=' * 75)
print('WINDOW-BY-WINDOW STATISTICS')
print('=' * 75)
print(stats_df.to_string(index=False))
print('=' * 75)
print()
print('Summary across all windows:')
summary_cols = ['n_nodes', 'n_edges', 'density', 'attack_rate', 'topo_anomalies']
print(stats_df[summary_cols].describe().round(4).to_string())

WINDOW-BY-WINDOW STATISTICS
 window_id  start_flow  end_flow  n_flows  n_nodes  n_edges  density  attack_rate  topo_anomalies  elapsed_sec
         0           0      5000     5000       39      156 0.105263       0.0010               1         0.54
         1        2500      7500     5000       36      138 0.109524       0.0008               2         0.58
         2        5000     10000     5000       37      141 0.105856       0.0016               0         0.62
         3        7500     12500     5000       39      147 0.099190       0.0014               0         0.66
         4       10000     15000     5000       41      150 0.091463       0.0020               0         0.70
         5       12500     17500     5000       39      147 0.099190       0.0024               0         0.73
         6       15000     20000     5000       35      141 0.118487       0.0004               0         0.77
         7       17500     22500     5000       37      143 0.107357       0.0004   

## 3.8 -- Suspicious node analysis

Identify highest-risk nodes by each topology metric.
This section directly supports the paper results section by showing
which nodes the delta signal flags that static centrality would miss.

Key finding from the report: `attack_ratio` is the strongest single
topology signal. `degree_delta` is the most powerful temporal feature
-- a delta of 25+ between consecutive windows indicates a scan starting,
invisible to static graphs.

In [10]:
def print_node_analysis(topo_all: pd.DataFrame) -> pd.DataFrame:
    """
    Print top nodes by each topology metric and return node averages.

    Demonstrates which nodes the delta signal flags that static
    metrics would miss -- directly supports paper results section.

    Parameters
    ----------
    topo_all : pd.DataFrame
        Concatenated topology features across all windows.

    Returns
    -------
    pd.DataFrame
        Per-node average features across all windows.
    """
    metric_cols = [
        'pagerank', 'betweenness', 'in_degree', 'out_degree',
        'clustering', 'flow_volume', 'attack_ratio',
        'degree_delta', 'volume_delta', 'topo_anomaly',
    ]
    available = [c for c in metric_cols if c in topo_all.columns]

    node_avg = (
        topo_all.groupby('node')[available]
        .mean()
        .reset_index()
    )

    print('=' * 65)
    print('SUSPICIOUS NODE ANALYSIS')
    print('=' * 65)

    print('Top 5 by out-degree (potential scanners / attackers):')
    cols = [c for c in ['node','out_degree','degree_delta','attack_ratio']
            if c in node_avg.columns]
    print(node_avg.nlargest(5, 'out_degree')[cols].to_string(index=False))

    print()
    print('Top 5 by in-degree (potential victims / DDoS targets):')
    cols = [c for c in ['node','in_degree','attack_ratio','flow_volume']
            if c in node_avg.columns]
    print(node_avg.nlargest(5, 'in_degree')[cols].to_string(index=False))

    print()
    print('Top 5 by betweenness (potential gateway / exfiltration nodes):')
    cols = [c for c in ['node','betweenness','attack_ratio','degree_delta']
            if c in node_avg.columns]
    print(node_avg.nlargest(5, 'betweenness')[cols].to_string(index=False))

    print()
    print('Top 5 by attack_ratio (strongest direct signal):')
    cols = [c for c in ['node','attack_ratio','out_degree','degree_delta']
            if c in node_avg.columns]
    print(node_avg.nlargest(5, 'attack_ratio')[cols].to_string(index=False))

    print()
    print('Top 5 by degree_delta (novel signal -- scan onset):')
    cols = [c for c in ['node','degree_delta','out_degree','attack_ratio']
            if c in node_avg.columns]
    print(node_avg.nlargest(5, 'degree_delta')[cols].to_string(index=False))

    print('=' * 65)
    return node_avg


topo_node_avg = print_node_analysis(topo_all)

SUSPICIOUS NODE ANALYSIS
Top 5 by out-degree (potential scanners / attackers):
      node  out_degree  degree_delta  attack_ratio
59.166.0.0        10.0           0.0           0.0
59.166.0.1        10.0           0.0           0.0
59.166.0.2        10.0           0.0           0.0
59.166.0.3        10.0           0.0           0.0
59.166.0.4        10.0           0.0           0.0

Top 5 by in-degree (potential victims / DDoS targets):
         node  in_degree  attack_ratio  flow_volume
149.171.126.0       10.0           0.0   1894061.34
149.171.126.1       10.0           0.0   1955742.06
149.171.126.2       10.0           0.0   2052193.00
149.171.126.3       10.0           0.0   1907545.60
149.171.126.4       10.0           0.0   1961418.16

Top 5 by betweenness (potential gateway / exfiltration nodes):
         node  betweenness  attack_ratio  degree_delta
149.171.126.3     0.019181           0.0      0.653333
149.171.126.2     0.016640           0.0      0.616000
149.171.126.4     

## 3.9 -- Build full IP-level graph for visualization

One static graph over the entire dataset for Gephi export and
submission visualization only. ML feature extraction uses the
per-window graphs built above.

In [11]:
def build_full_ip_graph(df: pd.DataFrame) -> nx.DiGraph:
    """
    Build a directed graph over the entire dataset using IP-level nodes.

    Used for visualization and Gephi export only.
    ML feature extraction uses the per-window sliding graphs.

    Parameters
    ----------
    df : pd.DataFrame
        Full graph data with src_ip, dst_ip, fwd_bytes, label.

    Returns
    -------
    nx.DiGraph
        Directed weighted graph with aggregated edge statistics.
    """
    agg = (
        df.groupby(['src_ip', 'dst_ip'])
        .agg(
            total_bytes  = ('fwd_bytes', 'sum'),
            flow_count   = ('fwd_bytes', 'count'),
            attack_flows = ('label',     'sum'),
        )
        .reset_index()
    )

    G = nx.DiGraph()
    for _, row in agg.iterrows():
        G.add_edge(
            str(row['src_ip']),
            str(row['dst_ip']),
            weight       = float(row['total_bytes']) + 1.0,
            flow_count   = int(row['flow_count']),
            is_malicious = int(row['attack_flows'] > 0),
        )

    log.info('Full graph: %d nodes, %d edges, density=%.4f',
             G.number_of_nodes(), G.number_of_edges(), nx.density(G))
    return G


G_full = build_full_ip_graph(graph_df)

print('Full IP graph:')
print('  Nodes   :', G_full.number_of_nodes())
print('  Edges   :', G_full.number_of_edges())
print('  Density : {:.4f}'.format(nx.density(G_full)))

gexf_path = CFG.DATA_GRAPH / 'network_graph.gexf'
nx.write_gexf(G_full, str(gexf_path))
log.info('Saved: network_graph.gexf')
print('  Saved  : network_graph.gexf')

08:48:51  INFO  Full graph: 44 nodes, 293 edges, density=0.1549
08:48:51  INFO  Saved: network_graph.gexf


Full IP graph:
  Nodes   : 44
  Edges   : 293
  Density : 0.1549
  Saved  : network_graph.gexf


## 3.10 -- Enrich flows with topology features

Merge per-node average topology features back onto the original flow records.
Each flow receives the average topology profile of its source IP and
destination IP across all windows it appeared in.

This produces the enriched feature matrix for Phase 4 hybrid models:
38 flow + DPI features + 9 topology features per source node
+ 9 topology features per destination node = 56 total features.

In [12]:
def enrich_flows_with_topology(
    graph_df: pd.DataFrame,
    topo_node_avg: pd.DataFrame,
) -> pd.DataFrame:
    """
    Merge per-node average topology features onto flow records.

    Each flow is assigned the average topology profile of its
    source IP and destination IP across all time windows.
    Flows with IPs not seen in any window receive zeros.

    Parameters
    ----------
    graph_df      : pd.DataFrame  Flow records from Phase 2.
    topo_node_avg : pd.DataFrame  Per-node average features.

    Returns
    -------
    pd.DataFrame
        Flow records enriched with src and dst topology features.
    """
    topo_cols = [
        'pagerank', 'betweenness', 'in_degree', 'out_degree',
        'clustering', 'flow_volume', 'attack_ratio',
        'degree_delta', 'volume_delta',
    ]
    available = [c for c in topo_cols if c in topo_node_avg.columns]

    # Source node topology
    src_topo = topo_node_avg[['node'] + available].copy()
    src_topo = src_topo.rename(columns=
        {'node': 'src_ip', **{c: f'src_topo_{c}' for c in available}}
    )

    # Destination node topology
    dst_topo = topo_node_avg[['node'] + available].copy()
    dst_topo = dst_topo.rename(columns=
        {'node': 'dst_ip', **{c: f'dst_topo_{c}' for c in available}}
    )

    enriched = graph_df.merge(src_topo, on='src_ip', how='left')
    enriched = enriched.merge(dst_topo, on='dst_ip', how='left')

    topo_added = [
        c for c in enriched.columns
        if c.startswith('src_topo_') or c.startswith('dst_topo_')
    ]
    enriched[topo_added] = enriched[topo_added].fillna(0)

    log.info(
        'Enriched: %d rows x %d columns (%d topology cols added)',
        *enriched.shape, len(topo_added),
    )
    return enriched


enriched_df = enrich_flows_with_topology(graph_df, topo_node_avg)

topo_added = [c for c in enriched_df.columns
              if c.startswith('src_topo_') or c.startswith('dst_topo_')]

print('Enriched flows shape    :', enriched_df.shape)
print('Topology columns added  :', len(topo_added))
print('Topology columns        :', topo_added)

08:49:04  INFO  Enriched: 2365636 rows x 31 columns (18 topology cols added)


Enriched flows shape    : (2365636, 31)
Topology columns added  : 18
Topology columns        : ['src_topo_pagerank', 'src_topo_betweenness', 'src_topo_in_degree', 'src_topo_out_degree', 'src_topo_clustering', 'src_topo_flow_volume', 'src_topo_attack_ratio', 'src_topo_degree_delta', 'src_topo_volume_delta', 'dst_topo_pagerank', 'dst_topo_betweenness', 'dst_topo_in_degree', 'dst_topo_out_degree', 'dst_topo_clustering', 'dst_topo_flow_volume', 'dst_topo_attack_ratio', 'dst_topo_degree_delta', 'dst_topo_volume_delta']


## 3.11 -- Visualizations for submission

Six plots saved to `outputs/plots/` at 200 DPI:
1. Temporal analysis: attack rate, graph density, anomaly count per window
2. Topology anomaly score distribution: benign vs attack
3. Full network topology graph (shell layout, subnet coloring)
4. Attack path subgraph (malicious edges only, PageRank sizing)
5. Top nodes by degree_delta (the novel signal)
6. Topology feature distributions: benign vs attack

In [13]:
plt.rcParams.update({
    'figure.dpi'       : 200,
    'font.size'        : 11,
    'axes.titlesize'   : 12,
    'axes.titleweight' : 'bold',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})


def save_figure(fig: plt.Figure, filename: str) -> None:
    """
    Save a figure to the plots directory at 200 DPI and close it.

    Parameters
    ----------
    fig      : plt.Figure  Figure to save.
    filename : str         Output filename.
    """
    path = CFG.PLOTS / filename
    fig.savefig(path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    log.info('Saved: %s', path)


def classify_node(ip: str) -> str:
    """
    Classify an IP address into a subnet category for visualization.

    Parameters
    ----------
    ip : str  IP address string.

    Returns
    -------
    str  Subnet category label.
    """
    ip = str(ip)
    if ip.startswith('59.166'):  return 'attacker'
    if ip.startswith('149.171'): return 'victim'
    if ip.startswith('175.45'):  return 'suspicious'
    if ip.startswith('10.40'):   return 'internal'
    return 'other'


SUBNET_COLOR = {
    'attacker'  : '#c0392b',
    'victim'    : '#2980b9',
    'suspicious': '#d35400',
    'internal'  : '#27ae60',
    'other'     : '#7f8c8d',
}


# Plot 1: Temporal analysis (3 panels)
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle('Dynamic Graph -- Temporal Analysis Across 50 Windows',
             fontsize=13, fontweight='bold')

axes[0].plot(stats_df['window_id'], stats_df['attack_rate'],
             color='#d7191c', linewidth=2)
axes[0].fill_between(stats_df['window_id'], stats_df['attack_rate'],
                     alpha=0.2, color='#d7191c')
axes[0].set_ylabel('Attack rate')
axes[0].set_title('Attack Rate per Window')
axes[0].grid(alpha=0.3)

axes[1].plot(stats_df['window_id'], stats_df['density'],
             color='#2c7bb6', linewidth=2)
axes[1].fill_between(stats_df['window_id'], stats_df['density'],
                     alpha=0.2, color='#2c7bb6')
axes[1].set_ylabel('Graph density')
axes[1].set_title('Topology Density per Window')
axes[1].grid(alpha=0.3)

axes[2].bar(stats_df['window_id'], stats_df['topo_anomalies'],
            color='#f4a11d', edgecolor='white', linewidth=0.5)
axes[2].set_ylabel('Anomaly count')
axes[2].set_title('Topology Anomalies Detected per Window (2-sigma rule)')
axes[2].set_xlabel('Window ID')
axes[2].grid(alpha=0.3)

plt.tight_layout()
save_figure(fig, 'phase3_temporal_analysis.png')


# Plot 2: Full network topology (shell layout)
nodes      = list(G_full.nodes())
categories = [classify_node(n) for n in nodes]
colors     = [SUBNET_COLOR[c] for c in categories]
pr_scores  = nx.pagerank(G_full, weight='weight')
sizes      = [max(80, min(1400, pr_scores.get(n, 0.01)*80000)) for n in nodes]

attackers  = sorted([n for n in nodes if classify_node(n) == 'attacker'])
victims    = sorted([n for n in nodes if classify_node(n) == 'victim'])
suspicious = sorted([n for n in nodes if classify_node(n) == 'suspicious'])
internal   = sorted([n for n in nodes if classify_node(n) == 'internal'])
others     = sorted([n for n in nodes if classify_node(n) == 'other'])
shells     = [s for s in [suspicious, attackers, victims, internal + others] if s]

try:
    pos = nx.shell_layout(G_full, nlist=shells)
except Exception:
    pos = nx.spring_layout(G_full, seed=CFG.SEED, k=2.0)

mal_edges = [(u,v) for u,v,d in G_full.edges(data=True) if d.get('is_malicious')]
ben_edges = [(u,v) for u,v,d in G_full.edges(data=True) if not d.get('is_malicious')]

fig, ax = plt.subplots(figsize=(18, 16))
ax.set_facecolor('#f8f9fa')
nx.draw_networkx_edges(G_full, pos, edgelist=ben_edges,
                       alpha=0.12, width=0.5, edge_color='#bdc3c7',
                       arrows=False, ax=ax)
nx.draw_networkx_edges(G_full, pos, edgelist=mal_edges,
                       alpha=0.80, width=2.0, edge_color='#e74c3c',
                       arrows=True, arrowsize=14,
                       connectionstyle='arc3,rad=0.1', ax=ax)
nx.draw_networkx_nodes(G_full, pos, node_color=colors, node_size=sizes,
                       alpha=0.92, ax=ax, edgecolors='white', linewidths=1.0)
labels_short = {n: '.'.join(str(n).split('.')[-2:]) for n in nodes}
nx.draw_networkx_labels(G_full, pos, labels=labels_short,
                        font_size=6, font_weight='bold', ax=ax)

seen_cats = set(categories)
legend_patches = [
    mpatches.Patch(color=col, label=cat)
    for cat, col in SUBNET_COLOR.items() if cat in seen_cats
]
legend_patches += [
    mpatches.Patch(color='#e74c3c', label='Malicious edge ({:,})'.format(len(mal_edges))),
    mpatches.Patch(color='#bdc3c7', label='Benign edge ({:,})'.format(len(ben_edges))),
]
ax.legend(handles=legend_patches, loc='lower left', fontsize=10, framealpha=0.9)
ax.set_title(
    'Full Network Topology -- NF-UNSW-NB15-v2\n'
    'Shell layout: subnets in concentric rings  |  '
    'Node size = PageRank  |  Color = subnet role',
    fontsize=13, fontweight='bold',
)
ax.axis('off')
save_figure(fig, 'phase3_full_topology.png')


# Plot 3: Attack path subgraph (malicious edges only)
mal_nodes = set()
for u, v in mal_edges:
    mal_nodes.add(u)
    mal_nodes.add(v)

if mal_nodes:
    MG = G_full.subgraph(list(mal_nodes)).copy()
    pr_mg = nx.pagerank(MG, weight='weight')
    pos_mg = nx.kamada_kawai_layout(MG, weight=None)

    mg_nodes = list(MG.nodes())
    mg_colors = [SUBNET_COLOR[classify_node(n)] for n in mg_nodes]
    mg_sizes  = [max(200, min(2000, pr_mg.get(n, 0.01)*100000))
                 for n in mg_nodes]

    fig, ax = plt.subplots(figsize=(14, 11))
    ax.set_facecolor('#f8f9fa')
    mal_only = [(u,v) for u,v,d in MG.edges(data=True) if d.get('is_malicious')]
    ben_only = [(u,v) for u,v,d in MG.edges(data=True) if not d.get('is_malicious')]

    if ben_only:
        nx.draw_networkx_edges(MG, pos_mg, edgelist=ben_only,
                               alpha=0.3, width=0.8, edge_color='#bdc3c7',
                               arrows=False, ax=ax)
    nx.draw_networkx_edges(MG, pos_mg, edgelist=mal_only,
                           alpha=0.85, width=2.5, edge_color='#e74c3c',
                           arrows=True, arrowsize=18,
                           connectionstyle='arc3,rad=0.15', ax=ax)
    nx.draw_networkx_nodes(MG, pos_mg, node_color=mg_colors,
                           node_size=mg_sizes, alpha=0.92, ax=ax,
                           edgecolors='white', linewidths=1.2)
    nx.draw_networkx_labels(MG, pos_mg, font_size=7,
                            font_weight='bold', ax=ax)

    # Annotate top 3 by PageRank
    top3 = sorted(pr_mg, key=pr_mg.get, reverse=True)[:3]
    role_map = {'attacker':'SCANNER','victim':'TARGET',
                'suspicious':'GATEWAY','internal':'INTERNAL','other':'HOST'}
    for rank, node in enumerate(top3, 1):
        if node in pos_mg:
            x, y = pos_mg[node]
            role  = role_map.get(classify_node(node), 'HOST')
            ax.annotate(
                '#{} {}\nPR={:.3f}'.format(rank, role, pr_mg[node]),
                xy=(x, y), xytext=(x + 0.12, y + 0.12),
                fontsize=8, color='darkred', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow',
                          edgecolor='darkred', alpha=0.9),
                arrowprops=dict(arrowstyle='->', color='darkred', lw=1.2)
            )

    seen2 = set(classify_node(n) for n in mg_nodes)
    leg2 = [
        mpatches.Patch(color=SUBNET_COLOR[c], label=c)
        for c in SUBNET_COLOR if c in seen2
    ]
    ax.legend(handles=leg2, loc='lower left', fontsize=9)
    ax.set_title(
        'Attack Path Subgraph -- Malicious Traffic Only\n'
        'Node size = PageRank  |  Gold labels = Top-3 high-risk nodes',
        fontsize=12, fontweight='bold',
    )
    ax.axis('off')
    save_figure(fig, 'phase3_attack_paths.png')


# Plot 4: Topology feature distributions benign vs attack
plot_topo = [c for c in
             ['degree_delta', 'volume_delta', 'attack_ratio',
              'out_degree', 'betweenness', 'pagerank']
             if c in topo_all.columns]

benign_topo = topo_all[topo_all['topo_anomaly'] == 0]
attack_topo = topo_all[topo_all['topo_anomaly'] == 1]

if len(plot_topo) >= 4:
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    fig.suptitle(
        'Topology Feature Distributions -- Normal vs Anomalous Nodes',
        fontsize=12, fontweight='bold',
    )
    axes = axes.flatten()

    for i, col in enumerate(plot_topo):
        q99 = topo_all[col].quantile(0.99)
        b_vals = benign_topo[col].clip(0, q99)
        a_vals = attack_topo[col].clip(0, q99)
        axes[i].hist(b_vals, bins=40, alpha=0.6, color='#2c7bb6',
                     density=True, label='Normal')
        if len(a_vals) > 0:
            axes[i].hist(a_vals, bins=40, alpha=0.6, color='#d7191c',
                         density=True, label='Anomaly')
        axes[i].set_title(col, fontsize=10)
        axes[i].set_xlabel('Value')
        axes[i].set_ylabel('Density')
        axes[i].legend(fontsize=8)

    plt.tight_layout()
    save_figure(fig, 'phase3_topo_distributions.png')

print('All plots saved to outputs/plots/')

08:49:28  INFO  Saved: C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\outputs\plots\phase3_temporal_analysis.png
08:49:29  INFO  Saved: C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\outputs\plots\phase3_full_topology.png
08:49:30  INFO  Saved: C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\outputs\plots\phase3_attack_paths.png
08:49:32  INFO  Saved: C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\outputs\plots\phase3_topo_distributions.png


All plots saved to outputs/plots/


## 3.12 -- Save all outputs

Save all Phase 3 artifacts needed by Phase 4.

In [14]:
def save_phase3_outputs(
    topo_all: pd.DataFrame,
    topo_node_avg: pd.DataFrame,
    stats_df: pd.DataFrame,
    enriched_df: pd.DataFrame,
) -> None:
    """
    Save all Phase 3 outputs to disk.

    Files written:
        data/graph/topo_features_dynamic.csv -- per-node features all windows
        data/graph/topo_node_avg.csv         -- per-node averages
        data/graph/window_stats.csv          -- per-window graph statistics
        data/processed/enriched_flows.csv    -- flows with topology features

    Parameters
    ----------
    topo_all      : pd.DataFrame  Topology features across all windows.
    topo_node_avg : pd.DataFrame  Per-node average features.
    stats_df      : pd.DataFrame  Per-window statistics.
    enriched_df   : pd.DataFrame  Flows enriched with topology.
    """
    topo_all.to_csv(
        CFG.DATA_GRAPH / 'topo_features_dynamic.csv', index=False
    )
    topo_node_avg.to_csv(
        CFG.DATA_GRAPH / 'topo_node_avg.csv', index=False
    )
    stats_df.to_csv(
        CFG.DATA_GRAPH / 'window_stats.csv', index=False
    )
    enriched_df.to_csv(
        CFG.DATA_PROCESSED / 'enriched_flows.csv', index=False
    )
    log.info('All Phase 3 outputs saved')


save_phase3_outputs(topo_all, topo_node_avg, stats_df, enriched_df)

print('=' * 60)
print('PHASE 3 COMPLETE')
print('=' * 60)

output_files = [
    (CFG.DATA_GRAPH     / 'topo_features_dynamic.csv', 'data/graph/'),
    (CFG.DATA_GRAPH     / 'topo_node_avg.csv',          'data/graph/'),
    (CFG.DATA_GRAPH     / 'window_stats.csv',            'data/graph/'),
    (CFG.DATA_GRAPH     / 'network_graph.gexf',          'data/graph/'),
    (CFG.DATA_PROCESSED / 'enriched_flows.csv',          'data/processed/'),
]
print('Files written:')
for path, location in output_files:
    if path.exists():
        print('  {:<42} {:>8.2f} MB  ({})'.format(
            path.name, path.stat().st_size / 1e6, location
        ))

print()
print('Topology summary:')
print('  Windows processed    :', len(stats_df))
print('  Topology records     :', len(topo_all))
print('  Unique nodes tracked :', topo_all['node'].nunique())
print('  Topo anomalies found :', int(topo_all['topo_anomaly'].sum()))
print('  Base features/node   : 7')
print('  Delta features/node  : 2 (degree_delta, volume_delta)')
print('  Total topo features  : 9')
print('  Topology cols in enriched_flows:',
      len([c for c in enriched_df.columns
           if c.startswith('src_topo_') or c.startswith('dst_topo_')]))
print()
print('Next: run Phase4_ML_SHAP.ipynb')

08:51:24  INFO  All Phase 3 outputs saved


PHASE 3 COMPLETE
Files written:
  topo_features_dynamic.csv                      0.13 MB  (data/graph/)
  topo_node_avg.csv                              0.01 MB  (data/graph/)
  window_stats.csv                               0.00 MB  (data/graph/)
  network_graph.gexf                             0.07 MB  (data/graph/)
  enriched_flows.csv                           628.23 MB  (data/processed/)

Topology summary:
  Windows processed    : 50
  Topology records     : 1557
  Unique nodes tracked : 42
  Topo anomalies found : 10
  Base features/node   : 7
  Delta features/node  : 2 (degree_delta, volume_delta)
  Total topo features  : 9
  Topology cols in enriched_flows: 18

Next: run Phase4_ML_SHAP.ipynb
